# Age Decile Survival Analysis — PBTA_RNA

**Purpose:** Determine whether age at diagnosis (grouped into deciles) is associated with overall (OS) and event-free (EFS) survival.
**Approach:** Kaplan-Meier curves with multi-group log-rank tests, globally and per cancer type.
**Data:** PBTA_RNA clinical data (patient + sample files).

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import chi2
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = "../PBTA_RNA"
PATIENT_FILE = f"{DATA_DIR}/data_clinical_patient_attributes.txt"
SAMPLE_FILE = f"{DATA_DIR}/data_clinical_sample_attributes.txt"

# --- Helpers (copied from survival_analysis.ipynb) ---
def read_patients():
    return pd.read_csv(PATIENT_FILE, sep="\t", header=4,
                       dtype={"AGE": float, "OS_MONTHS": float, "EFS_MONTHS": float})
def read_samples():
    return pd.read_csv(SAMPLE_FILE, sep="\t", header=4)

def clean_os(df):
    df["OS_STATUS"] = df["OS_STATUS"].str.strip()
    df["os_event"] = df["OS_STATUS"].apply(
        lambda x: 1 if pd.notna(x) and x.startswith("1:") else (0 if pd.notna(x) and x.startswith("0:") else np.nan))
    return df

def clean_efs(df):
    df["EFS_STATUS"] = df["EFS_STATUS"].str.strip()
    df["efs_event"] = df["EFS_STATUS"].apply(
        lambda x: 0 if pd.notna(x) and x == "0:No Event" else (1 if pd.notna(x) and x != "1:NA" else np.nan))
    return df

def kaplan_meier(times, events):
    d = pd.DataFrame({"t": times, "e": events}).dropna().sort_values("t")
    surv = 1.0; n = len(d); res = []
    for t, grp in d.groupby("t", sort=False):
        ne = int(grp["e"].sum())
        if ne > 0: surv *= (1 - ne / n)
        res.append({"t": t, "s": surv, "n": n, "ne": ne})
        n -= len(grp)
    return pd.DataFrame(res)

def add_km(fig, km, label, color, row=None, col=None):
    fig.add_trace(go.Scatter(
        x=km["t"], y=km["s"], mode="lines", name=label,
        line=dict(color=color, width=2, shape="hv"),
        legendgroup=label,
        hovertemplate=f"Time: %{{x}}<br>Survival: %{{y:.3f}}<extra>{label}</extra>"),
        row=row, col=col)
    return fig

def logrank_multi(groups):
    ng = len(groups)
    if ng < 2: return 1.0
    all_t = sorted(set(pd.concat([pd.Series(g[0].dropna()) for g in groups]).dropna()))
    if len(all_t) < 2: return 1.0
    O = np.zeros(ng); E = np.zeros(ng); V = np.zeros((ng, ng))
    for t in all_t:
        ar = np.array([(g[0] >= t).sum() for g in groups]); nr = ar.sum()
        if nr == 0: continue
        ev = np.array([((g[0] == t) & (g[1] == 1)).sum() for g in groups]); ot = ev.sum()
        if ot == 0: continue
        O += ev; E += ot * ar / nr
        if nr > 1:
            for i in range(ng):
                for j in range(ng):
                    if i == j: V[i, j] += ot * ar[i] / nr * (1 - ar[i] / nr) * (nr - ot) / (nr - 1)
                    else: V[i, j] -= ot * ar[i] / nr * ar[j] / nr * (nr - ot) / (nr - 1)
    try: return 1 - chi2.cdf((O - E) @ np.linalg.pinv(V) @ (O - E), ng - 1)
    except: return 1.0

print("Setup complete.")


Setup complete.


In [2]:
patients = read_patients()
samples = read_samples()
patients = clean_os(patients)
patients = clean_efs(patients)
merged = samples.merge(patients, on='PATIENT_ID', how='left')
print(f"Merged: {merged.shape[0]} rows, {merged['PATIENT_ID'].nunique()} patients")

# Create age deciles globally
age_valid = merged.dropna(subset=['AGE']).copy()
age_valid['age_decile_global'] = pd.qcut(age_valid['AGE'], 10, labels=False, duplicates='drop')
print(f"Global age decile ranges:")
for d in sorted(age_valid['age_decile_global'].unique()):
    sub = age_valid[age_valid['age_decile_global']==d]
    print(f"  Decile {d}: AGE=[{sub['AGE'].min():.1f}, {sub['AGE'].max():.1f}], n={len(sub)}")

# Target groups for per-cancer (n>=20 for OS and EFS)
MIN_SAMPLES = 20
os_data = merged.dropna(subset=['OS_MONTHS', 'os_event'])
per_group_n = os_data.groupby('CANCER_GROUP').size()
target_groups = per_group_n[per_group_n >= MIN_SAMPLES].index.tolist()
print(f"Groups for per-cancer analysis (n≥{MIN_SAMPLES} for OS): {len(target_groups)}")


Merged: 4312 rows, 2870 patients
Global age decile ranges:
  Decile 0: AGE=[0.0, 1.0], n=565
  Decile 1: AGE=[2.0, 2.0], n=286
  Decile 2: AGE=[3.0, 4.0], n=523
  Decile 3: AGE=[5.0, 6.0], n=470
  Decile 4: AGE=[7.0, 8.0], n=423
  Decile 5: AGE=[9.0, 10.0], n=412
  Decile 6: AGE=[11.0, 12.0], n=402
  Decile 7: AGE=[13.0, 14.0], n=345
  Decile 8: AGE=[15.0, 17.0], n=488
  Decile 9: AGE=[18.0, 73.0], n=317
Groups for per-cancer analysis (n≥20 for OS): 22


In [3]:
# Color scale: 10 shades from light blue to dark blue
decile_colors = px.colors.sequential.Blues[1:11]  # 10 shades
print(f"Using {len(decile_colors)} color shades for deciles")


Using 8 color shades for deciles


In [4]:
# Global: OS by age decile
df = merged.dropna(subset=['AGE', 'OS_MONTHS', 'os_event']).copy()
# Re-apply decile labels
df['age_decile'] = 'D' + pd.qcut(df['AGE'], 10, labels=False, duplicates='drop').astype(str)
n_deciles = df['age_decile'].nunique()
print(f"OS analysis: {len(df)} patients, {n_deciles} age deciles")
print(f"Age ranges per decile:")
for d in sorted(df['age_decile'].unique()):
    sub = df[df['age_decile']==d]
    print(f"  {d}: AGE=[{sub['AGE'].min():.1f}, {sub['AGE'].max():.1f}], n={len(sub)}, events={int(sub['os_event'].sum())}")

grps = [(df[df['age_decile']==v]['OS_MONTHS'], df[df['age_decile']==v]['os_event'])
        for v in sorted(df['age_decile'].unique())]
p = logrank_multi(grps)
print(f"Global log-rank: χ²(df={n_deciles-1}), p={p:.6f}")

fig = make_subplots(rows=2, cols=1, row_heights=[0.8, 0.2],
                    subplot_titles=(f'OS by Age Decile — Global log-rank p={p:.6f}', 'Risk table'))
for i, v in enumerate(sorted(df['age_decile'].unique())):
    sub = df[df['age_decile']==v]
    km = kaplan_meier(sub['OS_MONTHS'], sub['os_event'])
    fig = add_km(fig, km, f'{v} (n={len(sub)})', decile_colors[i % len(decile_colors)])
fig.update_layout(height=550, title='Overall Survival by Age Decile (Global)', template='plotly_white')
fig.update_yaxes(title_text='Survival Probability', row=1, col=1)
fig.update_xaxes(title_text='Months', row=2, col=1)
try:
    fig.show()
except:
    pass


OS analysis: 3475 patients, 10 age deciles
Age ranges per decile:
  D0: AGE=[0.0, 1.0], n=492, events=188
  D1: AGE=[2.0, 2.0], n=262, events=113
  D2: AGE=[3.0, 4.0], n=447, events=174


  D3: AGE=[5.0, 5.0], n=238, events=114
  D4: AGE=[6.0, 7.0], n=362, events=150
  D5: AGE=[8.0, 9.0], n=361, events=152
  D6: AGE=[10.0, 11.0], n=349, events=105
  D7: AGE=[12.0, 14.0], n=391, events=123
  D8: AGE=[15.0, 16.0], n=263, events=98
  D9: AGE=[17.0, 60.0], n=310, events=135


Global log-rank: χ²(df=9), p=0.000015


In [5]:
# Global: EFS by age decile
df = merged.dropna(subset=['AGE', 'EFS_MONTHS', 'efs_event']).copy()
# Re-apply decile labels
df['age_decile'] = 'D' + pd.qcut(df['AGE'], 10, labels=False, duplicates='drop').astype(str)
n_deciles = df['age_decile'].nunique()
print(f"EFS analysis: {len(df)} patients, {n_deciles} age deciles")
print(f"Age ranges per decile:")
for d in sorted(df['age_decile'].unique()):
    sub = df[df['age_decile']==d]
    print(f"  {d}: AGE=[{sub['AGE'].min():.1f}, {sub['AGE'].max():.1f}], n={len(sub)}, events={int(sub['efs_event'].sum())}")

grps = [(df[df['age_decile']==v]['EFS_MONTHS'], df[df['age_decile']==v]['efs_event'])
        for v in sorted(df['age_decile'].unique())]
p = logrank_multi(grps)
print(f"Global log-rank: χ²(df={n_deciles-1}), p={p:.6f}")

fig = make_subplots(rows=2, cols=1, row_heights=[0.8, 0.2],
                    subplot_titles=(f'EFS by Age Decile — Global log-rank p={p:.6f}', 'Risk table'))
for i, v in enumerate(sorted(df['age_decile'].unique())):
    sub = df[df['age_decile']==v]
    km = kaplan_meier(sub['EFS_MONTHS'], sub['efs_event'])
    fig = add_km(fig, km, f'{v} (n={len(sub)})', decile_colors[i % len(decile_colors)])
fig.update_layout(height=550, title='Event-Free Survival by Age Decile (Global)', template='plotly_white')
fig.update_yaxes(title_text='Survival Probability', row=1, col=1)
fig.update_xaxes(title_text='Months', row=2, col=1)
try:
    fig.show()
except:
    pass


EFS analysis: 3358 patients, 10 age deciles
Age ranges per decile:
  D0: AGE=[0.0, 1.0], n=486, events=342
  D1: AGE=[2.0, 2.0], n=262, events=196
  D2: AGE=[3.0, 4.0], n=432, events=311
  D3: AGE=[5.0, 5.0], n=214, events=146
  D4: AGE=[6.0, 7.0], n=329, events=211
  D5: AGE=[8.0, 9.0], n=347, events=225
  D6: AGE=[10.0, 11.0], n=340, events=194
  D7: AGE=[12.0, 14.0], n=377, events=214
  D8: AGE=[15.0, 16.0], n=263, events=148
  D9: AGE=[17.0, 60.0], n=308, events=178


Global log-rank: χ²(df=9), p=0.000000


In [6]:
# Per-cancer: OS by age decile (within each cancer group)
os_groups = [g for g in target_groups if 
             merged[(merged['CANCER_GROUP']==g) & merged['AGE'].notna() 
                    & merged['OS_MONTHS'].notna() & merged['os_event'].notna()].shape[0] >= 20]

results_os = []
n_groups = len(os_groups)
n_cols = min(3, n_groups)
n_rows = max(1, int(np.ceil(n_groups / n_cols)))

fig = make_subplots(rows=n_rows, cols=n_cols,
                    subplot_titles=[f'{g[:25]}' for g in os_groups],
                    vertical_spacing=0.08, horizontal_spacing=0.06)

idx = 0
for group in os_groups:
    gdata = merged[(merged['CANCER_GROUP']==group) & merged['AGE'].notna()
                   & merged['OS_MONTHS'].notna() & merged['os_event'].notna()].copy()
    if len(gdata) < 20: continue
    
    # Create deciles within this cancer group
    gdata['age_decile'] = 'D' + pd.qcut(gdata['AGE'], 10, labels=False, duplicates='drop').astype(str)
    if gdata['age_decile'].nunique() < 2: continue
    
    grps = [(gdata[gdata['age_decile']==v]['OS_MONTHS'], gdata[gdata['age_decile']==v]['os_event'])
            for v in sorted(gdata['age_decile'].unique())]
    p = logrank_multi(grps)
    results_os.append({'group': group, 'n': len(gdata), 'n_events': int(gdata['os_event'].sum()), 'p_value': p})
    
    row = idx // n_cols + 1
    col = idx % n_cols + 1
    
    for i, v in enumerate(sorted(gdata['age_decile'].unique())):
        sub = gdata[gdata['age_decile']==v]
        km = kaplan_meier(sub['OS_MONTHS'], sub['os_event'])
        fig = add_km(fig, km, f'{group[:15]} — {v} (n={len(sub)})', decile_colors[i % len(decile_colors)], row=row, col=col)
    
    fig.add_annotation(text=f'p={p:.4f}', xref=f'x{idx+1}', yref=f'y{idx+1}',
                       x=0.95, y=0.05, showarrow=False, font=dict(size=10))
    idx += 1

fig.update_layout(height=250*n_rows, title_text='OS by Age Decile per Cancer Group', template='plotly_white', showlegend=True)
try:
    fig.show()
except:
    pass

if results_os:
    print(f"\nPer-cancer OS by age decile results:")
    print(f"{'Cancer Group':45s} {'N':6s} {'Events':6s} {'p':8s}")
    print('-'*70)
    for r in sorted(results_os, key=lambda x: x['p_value']):
        sig = '✅' if r['p_value'] < 0.05 else '❌'
        print(f"{r['group']:45s} {r['n']:6d} {r['n_events']:6d} {r['p_value']:.4f}  {sig}")



Per-cancer OS by age decile results:
Cancer Group                                  N      Events p       
----------------------------------------------------------------------
High-grade glioma                                406    327 0.0000  ✅
Diffuse midline glioma                           368    356 0.0000  ✅
Choroid plexus tumor                              89     14 0.0000  ✅
Low-grade glioma                                 712     59 0.0000  ✅
Medulloblastoma                                  382    137 0.0000  ✅
Ewing sarcoma                                     34     19 0.0000  ✅
Diffuse hemispheric glioma                        26     24 0.0010  ✅
Atypical Teratoid Rhabdoid Tumor                 141     90 0.0013  ✅
Glial-neuronal tumor NOS                          47     12 0.0123  ✅
Sarcoma                                           45     23 0.0158  ✅
CNS Embryonal tumor                               24     14 0.0162  ✅
Chordoma                                          26

In [7]:
# Per-cancer: EFS by age decile (within each cancer group)
efs_groups = [g for g in target_groups if 
              merged[(merged['CANCER_GROUP']==g) & merged['AGE'].notna() 
                     & merged['EFS_MONTHS'].notna() & merged['efs_event'].notna()].shape[0] >= 20]

results_efs = []
n_groups = len(efs_groups)
n_cols = min(3, n_groups)
n_rows = max(1, int(np.ceil(n_groups / n_cols)))

fig = make_subplots(rows=n_rows, cols=n_cols,
                    subplot_titles=[f'{g[:25]}' for g in efs_groups],
                    vertical_spacing=0.08, horizontal_spacing=0.06)

idx = 0
for group in efs_groups:
    gdata = merged[(merged['CANCER_GROUP']==group) & merged['AGE'].notna()
                   & merged['EFS_MONTHS'].notna() & merged['efs_event'].notna()].copy()
    if len(gdata) < 20: continue
    
    # Create deciles within this cancer group
    gdata['age_decile'] = 'D' + pd.qcut(gdata['AGE'], 10, labels=False, duplicates='drop').astype(str)
    if gdata['age_decile'].nunique() < 2: continue
    
    grps = [(gdata[gdata['age_decile']==v]['EFS_MONTHS'], gdata[gdata['age_decile']==v]['efs_event'])
            for v in sorted(gdata['age_decile'].unique())]
    p = logrank_multi(grps)
    results_efs.append({'group': group, 'n': len(gdata), 'n_events': int(gdata['efs_event'].sum()), 'p_value': p})
    
    row = idx // n_cols + 1
    col = idx % n_cols + 1
    
    for i, v in enumerate(sorted(gdata['age_decile'].unique())):
        sub = gdata[gdata['age_decile']==v]
        km = kaplan_meier(sub['EFS_MONTHS'], sub['efs_event'])
        fig = add_km(fig, km, f'{group[:15]} — {v} (n={len(sub)})', decile_colors[i % len(decile_colors)], row=row, col=col)
    
    fig.add_annotation(text=f'p={p:.4f}', xref=f'x{idx+1}', yref=f'y{idx+1}',
                       x=0.95, y=0.05, showarrow=False, font=dict(size=10))
    idx += 1

fig.update_layout(height=250*n_rows, title_text='EFS by Age Decile per Cancer Group', template='plotly_white', showlegend=True)
try:
    fig.show()
except:
    pass

if results_efs:
    print(f"\nPer-cancer EFS by age decile results:")
    print(f"{'Cancer Group':45s} {'N':6s} {'Events':6s} {'p':8s}")
    print('-'*70)
    for r in sorted(results_efs, key=lambda x: x['p_value']):
        sig = '✅' if r['p_value'] < 0.05 else '❌'
        print(f"{r['group']:45s} {r['n']:6d} {r['n_events']:6d} {r['p_value']:.4f}  {sig}")



Per-cancer EFS by age decile results:
Cancer Group                                  N      Events p       
----------------------------------------------------------------------
Adamantinomatous Craniopharyngioma                98     66 0.0000  ✅
Diffuse midline glioma                           276    270 0.0000  ✅
High-grade glioma                                388    348 0.0000  ✅
Low-grade glioma                                 710    369 0.0000  ✅
Medulloblastoma                                  382    172 0.0000  ✅
Meningioma                                        87     57 0.0001  ✅
Ependymoma                                       296    207 0.0002  ✅
Diffuse hemispheric glioma                        26     25 0.0007  ✅
Choroid plexus tumor                              88     25 0.0008  ✅
Embryonal tumor with multilayer rosettes          22     20 0.0008  ✅
Schwannoma                                        46     29 0.0028  ✅
Glial-neuronal tumor NOS                          4

In [8]:
# Combine results
all_results = []
for r in results_os:
    all_results.append({'Phase': 'OS', 'Group': r['group'], 'N': r['n'], 'Events': r['n_events'], 'p_value': r['p_value'], 'Significant': '✅' if r['p_value'] < 0.05 else '❌'})
for r in results_efs:
    all_results.append({'Phase': 'EFS', 'Group': r['group'], 'N': r['n'], 'Events': r['n_events'], 'p_value': r['p_value'], 'Significant': '✅' if r['p_value'] < 0.05 else '❌'})

res_df = pd.DataFrame(all_results)
print(f"\n{'='*70}")
print(f"AGE DECILE SURVIVAL ANALYSIS — SUMMARY")
print(f"{'='*70}")
print(f"Total tests: {len(res_df)}")
print(f"Significant (p<0.05): {(res_df['p_value']<0.05).sum()}")
print(f"\nResults by cancer group:")
print(f"{'Group':45s} {'Phase':6s} {'N':6s} {'Events':6s} {'p':8s}")
print('-'*75)
for _, r in res_df.sort_values(['p_value', 'Phase']).iterrows():
    print(f"{r['Group']:45s} {r['Phase']:6s} {r['N']:6d} {r['Events']:6d} {r['p_value']:.4f}  {r['Significant']}")

# Save
import os
out_dir = 'age_deciles'
os.makedirs(out_dir, exist_ok=True)
res_df.to_csv(f'{out_dir}/age_deciles_results.csv', index=False)
print(f"\nSaved: {out_dir}/age_deciles_results.csv")



AGE DECILE SURVIVAL ANALYSIS — SUMMARY
Total tests: 44
Significant (p<0.05): 29

Results by cancer group:
Group                                         Phase  N      Events p       
---------------------------------------------------------------------------
High-grade glioma                             OS        406    327 0.0000  ✅
Diffuse midline glioma                        OS        368    356 0.0000  ✅
Adamantinomatous Craniopharyngioma            EFS        98     66 0.0000  ✅
Diffuse midline glioma                        EFS       276    270 0.0000  ✅
High-grade glioma                             EFS       388    348 0.0000  ✅
Choroid plexus tumor                          OS         89     14 0.0000  ✅
Low-grade glioma                              OS        712     59 0.0000  ✅
Low-grade glioma                              EFS       710    369 0.0000  ✅
Medulloblastoma                               OS        382    137 0.0000  ✅
Ewing sarcoma                                 OS